In [ ]:
from pathlib import Path
from typing import Union, List, Iterable
import re
import shutil

import numpy as np
import cv2
import pywt



def load_gray(path: Union[str, Path]) -> np.ndarray:
    """
    Load an image as float32 grayscale array (0-255).
    """
    p = Path(path)
    img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Could not read image: {p}")
    return img.astype(np.float32)


def make_ky_bandstop_mask(
    shape: tuple[int, int],
    ky_targets: list[float],
    *,
    sigma_ky: float = 0.005,   # cycles/pixel: thickness of the removed band
):
    """
    Build multiplicative FFT mask that removes energy at ky = ±k0 for ALL kx.
    Works in fftshifted coordinates.

    ky_targets: list of ky frequencies in cycles/pixel (positive numbers recommended).
    sigma_ky: Gaussian std dev of notch in ky (cycles/pixel).

    Returns:
      mask (float32), 1=keep, 0~remove at the ky bands
    """
    H, W = shape
    ky = np.fft.fftshift(np.fft.fftfreq(H, d=1.0))  # cycles/pixel
    KY = ky[:, None]  # shape (H, 1), broadcasts across W

    mask = np.ones((H, W), dtype=np.float32)

    for k0 in ky_targets:
        k0 = float(abs(k0))
        if k0 < 1e-12:
            continue

        # Gaussian "bumps" along ky only (applies for all kx)
        g_pos = np.exp(-0.5 * ((KY - k0) / sigma_ky) ** 2)
        g_neg = np.exp(-0.5 * ((KY + k0) / sigma_ky) ** 2)

        notch = (1.0 - g_pos) * (1.0 - g_neg)  # shape (H,1)
        mask *= notch.astype(np.float32)        # broadcasts to (H,W)

    return mask
    

def ky_bandstop_filter_image(
    img: np.ndarray,
    ky_targets: list[float],
    *,
    sigma_ky: float | None = None,
    detrend: bool = False,
):
    """
    Remove selected ky frequencies for all kx using FFT bandstop.
    Returns:
      filtered image (float32), mask (float32)
    """
    x = img.astype(np.float32)
    if detrend:
        x = x - np.mean(x)

    if sigma_ky is None:
        sigma_ky = 2.0 / x.shape[0]

    F = np.fft.fft2(x)
    Fshift = np.fft.fftshift(F)

    mask = make_ky_bandstop_mask(x.shape, ky_targets, sigma_ky=sigma_ky)
    Ffilt = Fshift * mask

    out = np.fft.ifft2(np.fft.ifftshift(Ffilt))
    out = np.real(out).astype(np.float32)
    return out, mask


def crop_image(img: np.ndarray, crop_box: tuple[int, int, int, int]) -> np.ndarray:
    """Crop an image given (x, y, w, h)."""
    x, y, w, h = crop_box
    return img[y:y + h, x:x + w]


def select_crop_region(image_path: Union[str, Path]) -> tuple[int, int, int, int]:
    """
    Opens an interactive OpenCV window to crop the baseline image.
    If a directory is passed, the first image in that directory is used.
    Returns (x, y, w, h) of the crop.
    """
    p = Path(image_path)
    if p.is_dir():
        firsts = _gather_frames(p)
        if not firsts:
            raise ValueError(f"No images found in baseline dir: {p}")
        image_path = firsts[0]
    else:
        image_path = p

    img = cv2.imread(str(image_path))
    if img is None:
        raise FileNotFoundError(image_path)

    cv2.namedWindow("Crop Baseline", cv2.WINDOW_NORMAL)
    cv2.resizeWindow("Crop Baseline", 1200, 800)

    print("Select crop region and press ENTER or SPACE when done. Press ESC to cancel.")
    r = cv2.selectROI("Crop Baseline", img, showCrosshair=True)
    cv2.destroyAllWindows()

    x, y, w, h = map(int, r)
    if w == 0 or h == 0:
        raise ValueError("No region selected.")
    print(f"Crop selected: x={x}, y={y}, w={w}, h={h}")
    return x, y, w, h


def select_exposure_region(image_path: Union[str, Path]) -> tuple[int, int, int, int]:
    """
    Select ROI used only for exposure matching (stable light source region).
    If a directory is passed, the first image in that directory is used.
    Returns (x, y, w, h).
    """
    p = Path(image_path)
    if p.is_dir():
        firsts = _gather_frames(p)
        if not firsts:
            raise ValueError(f"No images found in dir: {p}")
        image_path = firsts[0]
    else:
        image_path = p

    img = cv2.imread(str(image_path))
    if img is None:
        raise FileNotFoundError(image_path)

    win = "Crop Exposure ROI (Stable Light Source)"
    cv2.namedWindow(win, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(win, 1200, 800)

    print("Select EXPOSURE ROI and press ENTER or SPACE when done. Press ESC to cancel.")
    r = cv2.selectROI(win, img, showCrosshair=True)
    cv2.destroyAllWindows()

    x, y, w, h = map(int, r)
    if w == 0 or h == 0:
        raise ValueError("No exposure ROI selected.")
    print(f"Exposure ROI selected: x={x}, y={y}, w={w}, h={h}")
    return x, y, w, h



_LAST_INT_TIF = re.compile(r"(\d+)(?=\.(?:tif|tiff)$)", re.I)


def frame_index(path: Path) -> int:
    """
    Return trailing frame number for sorting.
    """
    m = _LAST_INT_TIF.search(path.name)
    if not m:
        raise ValueError(f"No trailing integer in filename: {path.name}")
    return int(m.group(1))


def _gather_frames(frames_dir: Path) -> List[Path]:
    """
    Collect frames with common image extensions.
    If any files end with a trailing integer before the extension,
    only include those and sort numerically.
    Otherwise, return all images sorted by name.
    """
    exts = ("*.tif", "*.tiff", "*.png", "*.jpg", "*.jpeg", "*.bmp")
    paths: List[Path] = []
    for ext in exts:
        paths.extend(frames_dir.glob(ext))
    if not paths:
        raise ValueError(f"No image files found in {frames_dir}")

    numeric = [p for p in paths if _LAST_INT_TIF.search(p.name)]
    if numeric:
        return sorted(numeric, key=frame_index)
    else:
        return sorted(paths, key=lambda p: p.name.lower())


def subtract_background_signed(frame: np.ndarray, baseline: np.ndarray) -> np.ndarray:
    """
    Signed background subtraction (keeps positive and negative differences).
    """
    return frame.astype(np.float32) - baseline.astype(np.float32)

'''
def subtract_background_absolute(frame: np.ndarray, baseline: np.ndarray) -> np.ndarray:
    """
    Absolute background subtraction.
    Returns the magnitude of the difference, discarding sign.
    """
    return np.abs(
        frame.astype(np.float32) - baseline.astype(np.float32)
    )
'''


def contrast_stretch(img: np.ndarray) -> np.ndarray:
    """
    Simple linear stretch so min -> 0 and max -> 255.
    Returns float32 in [0, 255].
    """
    min_val = float(img.min())
    max_val = float(img.max())
    if max_val - min_val < 1e-6:
        return np.full_like(img, 128.0, dtype=np.float32)
    scale = 255.0 / (max_val - min_val)
    stretched = (img - min_val) * scale
    return stretched.astype(np.float32)


def wvlt_denoise(frame: np.ndarray, threshold: float = 0.025) -> np.ndarray:
    """
    Wavelet denoising using soft thresholding of 2D wavelet coefficients.
    Uses rbio5.5 and preserves the wavedec2 coefficient structure.
    """
    frame = frame.astype(np.float32)
    wavelet = pywt.Wavelet("rbio5.5")

    # coeffs[0] = approximation, coeffs[1:] = list of (cH, cV, cD) tuples
    coeffs = pywt.wavedec2(frame, wavelet)

    # Option: threshold everything (approx + details)
    cA = coeffs[0]
    cA_thr = pywt.threshold(cA, threshold, mode="soft")

    detail_coeffs_thr = []
    for (cH, cV, cD) in coeffs[1:]:
        cH_t = pywt.threshold(cH, threshold, mode="soft")
        cV_t = pywt.threshold(cV, threshold, mode="soft")
        cD_t = pywt.threshold(cD, threshold, mode="soft")
        detail_coeffs_thr.append((cH_t, cV_t, cD_t))

    coeffs_thr = [cA_thr] + detail_coeffs_thr

    rec = pywt.waverec2(coeffs_thr, wavelet)
    rec = rec[:frame.shape[0], :frame.shape[1]]
    return rec.astype(np.float32)




def compute_mean_baseline(
    baseline_dir: Union[str, Path],
    crop_box: tuple[int, int, int, int],
    *,
    apply_ky_filter: bool = False, 
    ky_targets: list[float] | None = None, 
    ky_sigma: float | None = None,
) -> np.ndarray:
    """
    Load all baseline images, crop them, and return the mean baseline image
    """
    baseline_dir = Path(baseline_dir)
    paths = _gather_frames(baseline_dir)
    if not paths:
        raise ValueError(f"No baseline frames found in {baseline_dir}")

    mean_img = None
    count = 0

    for p in paths:
        img = load_gray(p)

        if apply_ky_filter:
            if not ky_targets:
                raise ValueError("apply_ky_filter=True requires ky_targets=[...]")
            img, _ = ky_bandstop_filter_image(img, ky_targets=ky_targets, sigma_ky=ky_sigma)
        
        img = crop_image(img, crop_box)

        if mean_img is None:
            mean_img = np.zeros_like(img, dtype=np.float32)

        mean_img += img
        count += 1

    mean_img /= float(count)
    print(f"Mean baseline computed from {count} frames")
    return mean_img



def write_video_from_frames(
    frames_dir: Union[str, Path],
    video_path: Union[str, Path] | None = None,
    *,
    fps: int = 60,
    codec: str = "mp4v",   # use "MJPG" if you want .avi instead
    pattern_exts: Iterable[str] = (".tif", ".tiff", ".png", ".jpg", ".jpeg", ".bmp"),
) -> Path:
    """
    Create a video from image frames in frames_dir
    - Sorts frames using the same trailing-integer logic as _gather_frames
    - Converts grayscale to BGR for better codec compatibility
    """
    frames_dir = Path(frames_dir)

    if video_path is None:
        video_path = frames_dir / f"{frames_dir.name}.mp4"
    else:
        video_path = Path(video_path)

    # Reuse your numeric sort behavior
    paths = [p for p in frames_dir.iterdir() if p.suffix.lower() in pattern_exts]
    if not paths:
        raise ValueError(f"No frames found in: {frames_dir}")

    try:
        paths = sorted(paths, key=frame_index)
    except Exception:
        paths = sorted(paths, key=lambda p: p.name.lower())

    # Read first frame for size
    first = cv2.imread(str(paths[0]), cv2.IMREAD_GRAYSCALE)
    if first is None:
        raise FileNotFoundError(paths[0])
    h, w = first.shape[:2]

    fourcc = cv2.VideoWriter_fourcc(*codec)
    writer = cv2.VideoWriter(str(video_path), fourcc, fps, (w, h), True)  # True = color

    if not writer.isOpened():
        print(f"[video] Primary codec '{codec}' failed for {video_path}, trying MJPG/.avi fallback")
        # Fallback to MJPG + .avi in the same folder
        fallback_path = video_path.with_suffix(".avi")
        fourcc_fallback = cv2.VideoWriter_fourcc(*"MJPG")
        writer = cv2.VideoWriter(str(fallback_path), fourcc_fallback, fps, (w, h), True)

        if not writer.isOpened():
            raise IOError(f"Could not open VideoWriter for either {video_path} or {fallback_path}")
        else:
            video_path = fallback_path
            print(f"[video] Using MJPG fallback at {fallback_path}")
    
    for i, p in enumerate(paths, 1):
        img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise FileNotFoundError(p)
        if img.shape[:2] != (h, w):
            raise ValueError(
                f"Frame size mismatch for {p.name}: expected {(w, h)}, got {(img.shape[1], img.shape[0])}"
            )

        bgr = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
        writer.write(bgr)

        if (i % 100) == 0:
            print(f"[video] {i} frames added...")

    writer.release()
    print(f"Video written: {video_path}")
    return video_path



def process_schlieren(
    schlieren_frame: Union[str, Path],
    baseline_dir: Union[str, Path],
    out_path: Union[str, Path],
    *,  
    baseline_single_path: Union[str, Path] | None = None,
    baseline_mode: str = "mean", # "mean" | "single" | "first" | "paired"
    use_existing_crop: tuple[int, int, int, int] | None = None,

    apply_ky_filter: bool = False,
    ky_targets: list[float] | None = None,
    ky_sigma: float | None = None,

    exposure_match_mode: str = "none", # "none" | "option1" | "option2" | "option3"
    exposure_eps: float = 1e-6,
    exposure_roi: tuple[int, int, int, int] | None=None,
    

    apply_contrast_stretch: bool = True,
    apply_wavelet_denoise: bool = False,
    wavelet_threshold: float = 1.0,

    make_video: bool = False,
    video_fps: int = 60,
    video_codec: str = "mp4v",
) -> tuple[np.ndarray, np.ndarray | None, tuple[int, int, int, int]]:

    baseline_dir = Path(baseline_dir)
    schlieren_frame = Path(schlieren_frame)
    out_path = Path(out_path)

    if not baseline_dir.exists():
        raise FileNotFoundError(f"Baseline directory does not exist: {baseline_dir}")

    # 1) crop box
    if use_existing_crop is None:
        crop_box = select_crop_region(baseline_dir)
    else:
        crop_box = use_existing_crop
        print(f"Reusing crop box: {crop_box}")

    baseline_mode = baseline_mode.lower().strip()

    exposure_match_mode = exposure_match_mode.lower().strip()
    if exposure_match_mode not in ("none", "option1", "option2", "option3"):
        raise ValueError("exposure_match_mode must be one of: 'none', 'option1', 'option2', 'option3'")

    if exposure_match_mode != "none" and baseline_mode != "paired":
        raise ValueError("exposure_match_mode is only supported when baseline_mode == 'paired'")

    # option3 needs a separate ROI (stable light source) for computing alpha
    if exposure_match_mode == "option3":
        if exposure_roi is None:
            exposure_roi = select_exposure_region(baseline_dir)
        else:
            print(f"Reusing exposure ROI: {exposure_roi}")

    # Hold either a single baseline image, or paired baseline paths
    baseline_img: np.ndarray | None = None
    baseline_frames: List[Path] | None = None

    # 2) baseline setup
    if baseline_mode == "mean":
        baseline_img = compute_mean_baseline(
            baseline_dir,
            crop_box,
            apply_ky_filter=apply_ky_filter,
            ky_targets=ky_targets,
            ky_sigma=ky_sigma,
        )

    elif baseline_mode == "first":
        first_path = _gather_frames(baseline_dir)[0]
        b = load_gray(first_path)
        if apply_ky_filter:
            if not ky_targets:
                raise ValueError("apply_ky_filter=True requires ky_targets=[...]")
            b, _ = ky_bandstop_filter_image(b, ky_targets=ky_targets, sigma_ky=ky_sigma)
        baseline_img = crop_image(b, crop_box)
        print(f"Using FIRST baseline frame: {first_path.name}")

    elif baseline_mode == "single":
        if baseline_single_path is None:
            raise ValueError("baseline_mode='single' requires baseline_single_path=...")
        bpath = Path(baseline_single_path)
        if not bpath.exists():
            raise FileNotFoundError(f"Baseline single frame not found: {bpath}")

        b = load_gray(bpath)
        if apply_ky_filter:
            if not ky_targets:
                raise ValueError("apply_ky_filter=True requires ky_targets=[...]")
            b, _ = ky_bandstop_filter_image(b, ky_targets=ky_targets, sigma_ky=ky_sigma)
        baseline_img = crop_image(b, crop_box)
        print(f"Using SINGLE baseline frame: {bpath.name}")

    elif baseline_mode == "paired":
        # Only meaningful for folder-of-frames mode
        if not schlieren_frame.is_dir():
            raise ValueError("baseline_mode='paired' requires schlieren_frame to be a directory of frames.")
        baseline_frames = _gather_frames(baseline_dir)

    else:
        raise ValueError("baseline_mode must be one of: 'mean', 'single', 'first', 'paired'")

    # ---------------- SINGLE FRAME MODE ---------------- #
    if schlieren_frame.is_file():
        if baseline_mode == "paired":
            raise ValueError("baseline_mode='paired' only works when schlieren_frame is a directory.")

        frame = load_gray(schlieren_frame)
        if apply_ky_filter:
            if not ky_targets:
                raise ValueError("apply_ky_filter=True requires ky_targets=[...]")
            frame, _ = ky_bandstop_filter_image(frame, ky_targets=ky_targets, sigma_ky=ky_sigma)

        frame_cropped = crop_image(frame, crop_box)

        assert baseline_img is not None
        if frame_cropped.shape != baseline_img.shape:
            raise ValueError(f"Shape mismatch after crop: frame {frame_cropped.shape}, baseline {baseline_img.shape}")

        diff = subtract_background_signed(frame_cropped, baseline_img)

        if apply_wavelet_denoise:
            diff = wvlt_denoise(diff, threshold=wavelet_threshold)

        out_path.parent.mkdir(parents=True, exist_ok=True)

        if apply_contrast_stretch:
            proc_uint8 = np.clip(contrast_stretch(diff), 0, 255).astype(np.uint8)
            if not cv2.imwrite(str(out_path), proc_uint8):
                raise IOError(f"Failed to write output frame to: {out_path}")
            return diff, proc_uint8, crop_box
        else:
            save_path = out_path if out_path.suffix else out_path.with_suffix(".tiff")
            if not cv2.imwrite(str(save_path), diff.astype(np.float32)):
                raise IOError(f"Failed to write float32 diff frame to: {save_path}")
            return diff, None, crop_box

    # ---------------- BATCH MODE ---------------- #
    if schlieren_frame.is_dir():
        frames_dir = schlieren_frame
        schlieren_frames = _gather_frames(frames_dir)
        if not schlieren_frames:
            raise ValueError(f"No schlieren frames found in {frames_dir}")

        out_dir = out_path
        out_dir.mkdir(parents=True, exist_ok=True)

        print(f"Batch processing {len(schlieren_frames)} frames from {frames_dir} into {out_dir}")

        # Paired baseline validation
        if baseline_mode == "paired":
            assert baseline_frames is not None
            if len(baseline_frames) != len(schlieren_frames):
                raise ValueError(
                    f"paired mode requires equal counts: baseline={len(baseline_frames)} "
                    f"schlieren={len(schlieren_frames)}"
                )

        s_ref: float | None = None
        if exposure_match_mode == "option2":
            s_means = []
            for s_path in schlieren_frames:
                s_tmp = load_gray(s_path)
                if apply_ky_filter:
                    if not ky_targets:
                        raise ValueError("apply_ky_filter=True requires ky_targets=[...]")
                    s_tmp, _ = ky_bandstop_filter_image(s_tmp, ky_targets=ky_targets, sigma_ky=ky_sigma)

                s_tmp = crop_image(s_tmp, crop_box)
                s_means.append(float(s_tmp.mean()))

            s_ref = float(np.mean(s_means))
            print(f"[exposure option2] s_ref (mean of per-frame means) = {s_ref:.6f}")

        last_diff: np.ndarray | None = None
        last_proc: np.ndarray | None = None

        for i in range(len(schlieren_frames)):
            s_path = schlieren_frames[i]
            s = load_gray(s_path)

            if apply_ky_filter:
                if not ky_targets:
                    raise ValueError("apply_ky_filter=True requires ky_targets=[...]")
                s, _ = ky_bandstop_filter_image(s, ky_targets=ky_targets, sigma_ky=ky_sigma)

            s_crop = crop_image(s, crop_box)

            if baseline_mode == "paired":
                b_path = baseline_frames[i]  # type: ignore[index]
                b = load_gray(b_path)

                if apply_ky_filter:
                    b, _ = ky_bandstop_filter_image(b, ky_targets=ky_targets, sigma_ky=ky_sigma)

                b_crop = crop_image(b, crop_box)
                if s_crop.shape != b_crop.shape:
                    raise ValueError(
                        f"Shape mismatch after crop at i={i}: schlieren {s_crop.shape}, baseline {b_crop.shape} "
                        f"({s_path.name} vs {b_path.name})"
                    )
                
                if exposure_match_mode == "none":
                    diff = subtract_background_signed(s_crop, b_crop)

                else:
                    # Choose which pixels define s_mean and b_mean (for alpha)
                    if exposure_match_mode == "option1":
                        # option1: alpha_j computed from the schlieren crop itself
                        s_mean = float(s_crop.mean())
                        b_mean = float(b_crop.mean())

                    elif exposure_match_mode == "option3":
                        # option3: alpha_j computed from a stable-light-source ROI on the full frames
                        assert exposure_roi is not None
                        s_roi = crop_image(s, exposure_roi)
                        b_roi = crop_image(b, exposure_roi)
                        s_mean = float(s_roi.mean())
                        b_mean = float(b_roi.mean())

                    else:
                        # option2: alpha_j = s_ref / b_j (b_j taken from the schlieren crop)
                        assert s_ref is not None
                        s_mean = s_ref
                        b_mean = float(b_crop.mean())

                    # Compute alpha safely
                    if abs(b_mean) < exposure_eps:
                        alpha = 1.0
                    else:
                        alpha = float(s_mean / b_mean)

                    # Apply alpha to baseline crop during subtraction
                    diff = s_crop.astype(np.float32) - (alpha * b_crop.astype(np.float32))
                    
            else:
                assert baseline_img is not None
                if s_crop.shape != baseline_img.shape:
                    raise ValueError(
                        f"Shape mismatch after crop for {s_path.name}: frame {s_crop.shape}, baseline {baseline_img.shape}"
                    )
                diff = subtract_background_signed(s_crop, baseline_img)

            if apply_wavelet_denoise:
                diff = wvlt_denoise(diff, threshold=wavelet_threshold)

            '''
            HERE IS CONTRAST STRETCH FOR BATCH
            '''
            if apply_contrast_stretch:
                proc_uint8 = np.clip(contrast_stretch(diff), 0, 255).astype(np.uint8)
                save_path = out_dir / f"{s_path.stem}_proc.tif"
                if not cv2.imwrite(str(save_path), proc_uint8):
                    raise IOError(f"Failed to write output frame to: {save_path}")
                last_proc = proc_uint8
            else:
                save_path = out_dir / f"{s_path.stem}_diff.tiff"
                if not cv2.imwrite(str(save_path), diff.astype(np.float32)):
                    raise IOError(f"Failed to write float32 diff frame to: {save_path}")
                last_proc = None

            last_diff = diff

            if ((i + 1) % 25 == 0) or (i + 1 == len(schlieren_frames)):
                print(f"[{i+1}/{len(schlieren_frames)}] processed {s_path.name}")

        print(f"Finished batch. {len(schlieren_frames)} frames written to {out_dir}")

        if make_video:
            ext = ".mp4" if video_codec.lower() != "mjpg" else ".avi"
            video_path = write_video_from_frames(
                out_dir,
                video_path=out_dir / f"{out_dir.name}{ext}",
                fps=video_fps,
                codec=video_codec,
            )
            print(f"Video saved at: {video_path}")

        return last_diff, last_proc, crop_box

    raise FileNotFoundError(f"Schlieren path is neither a file nor directory: {schlieren_frame}")         

In [ ]:
"""
#SINGLE-FRAME 




baseline_folder = r"G:\Shared drives\Shumlak Lab\Diagnostics\Schlieren\ZaPHD Tests\250930 Canned Air\baseline_10"
schlieren_frame = r"G:\Shared drives\Shumlak Lab\Diagnostics\Schlieren\ZaPHD Tests\250930 Canned Air\jet_10\jet_10 027.tif"
out_frame = r"G:\Shared drives\Shumlak Lab\Diagnostics\Schlieren\ZaPHD Tests\250930 Canned Air\jet_10 027 processed.tif"

raw_diff, stretched_img, crop_box = process_schlieren(
    schlieren_frame,      
    baseline_folder,
    out_frame,            
    apply_contrast_stretch=True,
    apply_wavelet_denoise=True,
)

"""

In [ ]:
"""
BATCH
"""
#crop = 74, 254, 742, 100
baseline_folder = r"C:\Users\ahada\OneDrive\University of Washington\Z-Pinch Fusion\Schlieren post processing\Schlieren data\260311\260311_background"
schlieren_folder = r"C:\Users\ahada\OneDrive\University of Washington\Z-Pinch Fusion\Schlieren post processing\Schlieren data\260311\260311_canned_air"
out_folder = r"C:\Users\ahada\OneDrive\University of Washington\Z-Pinch Fusion\Schlieren post processing\Schlieren data\260311\260311_canned_air PROCESSED (paired option 3 cropped alpha)"
#baseline_path = r"C:\Users\ahada\OneDrive\University of Washington\Z-Pinch Fusion\Schlieren post processing\Schlieren data\260217\1us_1000fps_background_realign\1us_1000fps_background_realign 002.tif"

raw_diff, stretched_img, crop_box = process_schlieren(
    schlieren_folder,      
    baseline_folder,
    out_folder, 
    #baseline_single_path=baseline_path,
    baseline_mode = "paired",    
    use_existing_crop=None,

    apply_ky_filter=False,
    ky_targets=[0.25, 0.375],
    ky_sigma=None,

    exposure_match_mode = "none",
    
    
    apply_contrast_stretch=True,
    apply_wavelet_denoise=True,
    make_video=True,      
    video_fps=30,        
    video_codec="MJPG",   
)


In [ ]:
x=74, y=254, w=742, h=100

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

def show_baseline_comparison(
    baseline_dir,
    single_baseline_path,
    *,
    crop_box=None,
    pick_crop_interactively=False,
):
    baseline_dir = Path(baseline_dir)
    single_baseline_path = Path(single_baseline_path)



    # ---- choose crop ----
    if pick_crop_interactively:
        crop_box = select_crop_region(baseline_dir)

    if crop_box is None:
        raise ValueError("Provide crop_box or enable interactive crop.")

    # ---- compute averaged baseline ----
    mean_baseline = compute_mean_baseline(baseline_dir, crop_box)

    # ---- load single baseline ----
    single = load_gray(single_baseline_path)
    vmin = single.min()
    vmax = single.max()

    single_c = crop_image(single, crop_box)

    if single_c.shape != mean_baseline.shape:
        raise ValueError("Shape mismatch after cropping.")

    # ---- difference ----
    diff = subtract_background_signed(single_c, mean_baseline)

    # stretch diff for visualization only
    diff_view = contrast_stretch(diff)

    # ---- metrics ----
    print("Metrics:")
    print(" mean diff:", diff.mean())
    print(" std diff :", diff.std())
    print(" RMS diff :", np.sqrt(np.mean(diff**2)))

    # ---- display ----
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow(single_c, cmap="gray")
    axes[0].set_title("Single Baseline")
    axes[0].axis("off")

    axes[1].imshow(mean_baseline, cmap="gray")
    axes[1].set_title("Averaged Baseline")
    axes[1].axis("off")

    axes[2].imshow(diff, cmap="gray")
    axes[2].set_title("Single − Mean (Contrast Stretched)")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

    return single_c, mean_baseline, diff

In [ ]:
crop = (74, 254, 742, 100)

single, mean, diff = show_baseline_comparison(
    r"C:\Users\ahada\OneDrive\University of Washington\Z-Pinch Fusion\Schlieren post processing\Schlieren data\260217\1us_1000fps_background",
    r"C:\Users\ahada\OneDrive\University of Washington\Z-Pinch Fusion\Schlieren post processing\Schlieren data\260219\260219009\FILTERED.tif",
    crop_box=crop
    #pick_crop_interactively=True
)